In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection/

/content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection


In [ ]:
import numpy as np
BASE_DIR = "./dataset_windows20/"

X_train = np.load(BASE_DIR + "train/X.npy")
y_train = np.load(BASE_DIR + "train/y_bin.npy")
y_ch_train = np.load(BASE_DIR + "train/y_ch.npy")

X_val = np.load(BASE_DIR + "val/X.npy")
y_val = np.load(BASE_DIR + "val/y_bin.npy")
y_ch_val = np.load(BASE_DIR + "val/y_ch.npy")

X_test = np.load(BASE_DIR + "test/X.npy")
y_test = np.load(BASE_DIR + "test/y_bin.npy")
y_ch_test = np.load(BASE_DIR + "test/y_ch.npy")

In [ ]:

print(y_ch_train.shape)

(279986, 15)


In [ ]:
X_train = X_train[:, :, 1:]
X_val   = X_val[:, :, 1:]
X_test  = X_test[:, :, 1:]

print(X_train.shape)


(279986, 20, 16)


In [ ]:
X_train = X_train[:, :, 0:-1]
X_val   = X_val[:, :, 0:-1]
X_test  = X_test[:, :, 0:-1]
print(X_train.shape)

(279986, 20, 15)


In [ ]:
## Preprocessing

In [ ]:
X_train_nom = X_train[y_train == 0]   # solo nominal

mean_feat = X_train_nom.mean(axis=(0,1))   # (F,)
std_feat  = X_train_nom.std(axis=(0,1)) + 1e-8


In [ ]:
def scale_windows(X, mean, std):
    return (X - mean[None, None, :]) / std[None, None, :]

X_train_s = scale_windows(X_train, mean_feat, std_feat)
X_val_s   = scale_windows(X_val,   mean_feat, std_feat)
X_test_s  = scale_windows(X_test,  mean_feat, std_feat)


In [ ]:
import pywt
import numpy as np

def compute_dwt_windows(X, wavelet="db4", level=1):
    """
    X: (N_windows, window_size, n_channels) o (N_windows, window_size)
    return:
        A: Approximation coefficients (low-frequency)
        D: Detail coefficients (high-frequency, concatenated)
    """
    if X.ndim == 2:
        X = X[..., np.newaxis]

    N, W, F = X.shape

    A_list = []
    D_list = []

    for i in range(N):
        A_ch = []
        D_ch = []

        for ch in range(F):
            coeffs = pywt.wavedec(X[i, :, ch], wavelet, level=level)

            A = coeffs[0]                  # Approximation
            D = np.concatenate(coeffs[1:]) # All detail levels

            A_ch.append(A)
            D_ch.append(D)

        A_list.append(np.stack(A_ch, axis=1))
        D_list.append(np.stack(D_ch, axis=1))

    return np.array(A_list), np.array(D_list)
# A_windows, D_windows = compute_dwt_windows(X)
# print("A:", A_windows.shape)
# print("D:", D_windows.shape)
def adaptive_threshold_detector_HF(window, thresholds):
    win_max = np.max(window, axis=0)
    return int(np.any(win_max > thresholds))
def adaptive_threshold_detector_LF(window, thresholds):
    win_min = np.min(window, axis=0)
    return int(np.any(win_min < thresholds))
def compute_thresholds(feature_windows, y_bin, k=2.0, mode="high"):
    # feature_windows: (N, coeffs, F)
    nominal = feature_windows[y_bin == 0]

    vals = np.max(nominal, axis=1) if mode == "high" else np.min(nominal, axis=1)

    mu = np.mean(vals, axis=0)
    sigma = np.std(vals, axis=0)

    if mode == "high":
        return mu + k * sigma
    else:
        return mu - k * sigma



In [ ]:
A_train, D_train = compute_dwt_windows(X_train_s)
A_val,   D_val   = compute_dwt_windows(X_val_s)
A_test,  D_test  = compute_dwt_windows(X_test_s)


In [ ]:
print(D_train.shape)
# (N_samples, n_coeffs, n_channels)


(279986, 13, 15)


In [ ]:
mean = np.mean(D_train, axis=(0,1), keepdims=True)
std  = np.std(D_train, axis=(0,1), keepdims=True) + 1e-8

D_train_n = (D_train - mean) / std
D_val_n   = (D_val   - mean) / std
D_test_n  = (D_test  - mean) / std


In [ ]:
def compute_energy_features(D):
    return np.sum(D**2, axis=1)

E_train = compute_energy_features(D_train_n)
E_val   = compute_energy_features(D_val_n)
E_test  = compute_energy_features(D_test_n)

print(E_train.shape)
# (N, 15)

(279986, 15)


In [ ]:
print(y_ch_train.shape)

(279986, 15)


In [ ]:
import tensorflow as tf
model = tf.keras.Sequential([
    tf.keras.layers.Dense(16, activation='relu', input_shape=(15,)),
    tf.keras.layers.Dense(15, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(E_train, y_ch_train,
          validation_data=(E_val, y_ch_val),
          epochs=20)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - accuracy: 0.2462 - loss: 0.9459 - val_accuracy: 0.3320 - val_loss: 0.1788
Epoch 2/20
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.3226 - loss: 0.1579 - val_accuracy: 0.3189 - val_loss: 0.1673
Epoch 3/20
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 19s 2ms/step - accuracy: 0.3254 - loss: 0.1413 - val_accuracy: 0.3262 - val_loss: 0.1337
Epoch 4/20
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 19s 2ms/step - accuracy: 0.3256 - loss: 0.1293 - val_accuracy: 0.3537 - val_loss: 0.1145
Epoch 5/20
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.3419 - loss: 0.1156 - val_accuracy: 0.3581 - val_loss: 0.1085
Epoch 6/20
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - accuracy: 0.3500 - loss: 0.1067 - val_accuracy: 0.3608 - val_loss: 0.0935
Epoch 7/20
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.3517 - loss: 0.1007 - val_accuracy: 0.3622 - val_loss: 0.0915
Epoch 8/20
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 18s 2ms/step - accuracy: 0.3532 - loss: 0

In [ ]:
y_pred = np.zeros_like(y_ch_test)

# Predict probabilities for all channels at once
all_channel_probs = model.predict(E_test)

for ch in range(15):
    # Extract probabilities for the current channel
    prob = all_channel_probs[:, ch]

    y_pred[:,ch] = (prob > 0.3).astype(int)   # threshold ajustable

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_ch_test,
    y_pred,
    target_names=[f"ch{i}" for i in range(15)],
    digits=4
))

              precision    recall  f1-score   support

         ch0     0.9574    0.7480    0.8399      2286
         ch1     0.8341    0.7442    0.7866      2330
         ch2     0.6899    0.7351    0.7118      2582
         ch3     0.7605    0.6618    0.7077      2629
         ch4     0.8440    0.7739    0.8074      2362
         ch5     0.8759    0.7441    0.8046      2704
         ch6     0.9394    0.6055    0.7363      3072
         ch7     0.9591    0.6055    0.7424      2710
         ch8     0.9523    0.6187    0.7501      2869
         ch9     0.9096    0.5439    0.6808      3015
        ch10     0.8983    0.5268    0.6641      2633
        ch11     0.9097    0.5892    0.7152      2975
        ch12     0.9704    0.7151    0.8234      3166
        ch13     0.8963    0.8060    0.8487      2670
        ch14     0.9739    0.7293    0.8340      2150

   micro avg     0.8828    0.6715    0.7628     40153
   macro avg     0.8914    0.6765    0.7635     40153
weighted avg     0.8932   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │           255 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,535 (6.00 KB)

 Trainable params: 511 (2.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,024 (4.00 KB)